# Task 14 - DLinear Forecasting

Notebook này chạy đủ 3 cấu hình Bảng 3: `DLinear_RawSales`, `DLinear_TimesNet` và `DLinear_DLinear`.

- Commit bắt buộc: `8af694a09a5a427d1d50648d348c022565a781f2`
- Seed: `2025`
- GPU: khuyến nghị NVIDIA T4 hoặc GPU tương đương
- Dữ liệu: `train.parquet`, `eval.parquet`, `demand_timesnet.parquet`, `demand_dlinear.parquet`

In [ ]:
# Mount Google Drive and verify the runtime.
from google.colab import drive
drive.mount('/content/drive')

import os, glob, subprocess, json
import torch

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Task 14 cần GPU. Hãy chọn Runtime type = GPU (khuyến nghị T4).')

In [ ]:
# Clone repository and pin the exact commit.
import os, shutil, subprocess

os.chdir('/content')
repo = '/content/frn-50k-baseline'
if not os.path.isdir(repo):
    subprocess.run([
        'git', 'clone', 'https://github.com/Dingdong-Inc/frn-50k-baseline.git', repo
    ], check=True)

subprocess.run(['git', '-C', repo, 'checkout', '--detach', '8af694a'], check=True)

commit = subprocess.run(
    ['git', '-C', repo, 'rev-parse', 'HEAD'],
    check=True, capture_output=True, text=True
).stdout.strip()
print(commit)
assert commit == '8af694a09a5a427d1d50648d348c022565a781f2'

In [ ]:
# Install the baseline dependencies.
!pip -q install --upgrade numpy pandas pyarrow datasets scikit-learn tqdm tensorboard einops
!pip -q install --upgrade torch torchvision

In [ ]:
# Find or copy the four required Parquet files.
import glob, os, shutil, hashlib

search_roots = ['/content', '/content/drive/MyDrive']
names = ['train.parquet', 'eval.parquet', 'demand_timesnet.parquet', 'demand_dlinear.parquet']
paths = {}

for name in names:
    found = []
    for root in search_roots:
        found.extend(glob.glob(os.path.join(root, '**', name), recursive=True))
    found = list(dict.fromkeys(found))
    if not found:
        raise FileNotFoundError(f'Missing {name}. Upload/copy it into Google Drive or /content.')
    if len(found) > 1:
        print(f'Multiple copies of {name}; using:', found[0])
    paths[name] = found[0]

data_dir = '/content/task14_data'
os.makedirs(data_dir, exist_ok=True)
for name, source in paths.items():
    destination = os.path.join(data_dir, name)
    if os.path.abspath(source) != os.path.abspath(destination):
        shutil.copy2(source, destination)
    print(name, os.path.getsize(destination), hashlib.sha256(open(destination, 'rb').read()).hexdigest())

TRAIN_PATH = os.path.join(data_dir, 'train.parquet')
EVAL_PATH = os.path.join(data_dir, 'eval.parquet')
TIMESNET_PATH = os.path.join(data_dir, 'demand_timesnet.parquet')
DLINEAR_PATH = os.path.join(data_dir, 'demand_dlinear.parquet')

In [ ]:
# Patch the author code to read local train/eval Parquet files.
from pathlib import Path

dlinear_dir = Path('/content/frn-50k-baseline/demand_forecasting/DLinear')
main_py = dlinear_dir / 'main.py'
main_text = main_py.read_text(encoding='utf-8')
old = "parser.add_argument('--data_path', type=str, default=None, help='data file')\n"
new = old + "parser.add_argument('--train_path', type=str, default=None, help='local train parquet file')\n" + "parser.add_argument('--eval_path', type=str, default=None, help='local eval parquet file')\n"
if "--train_path" not in main_text:
    main_text = main_text.replace(old, new)
    main_py.write_text(main_text, encoding='utf-8')

exp_py = dlinear_dir / 'exp' / 'exp_main.py'
exp_text = exp_py.read_text(encoding='utf-8')
exp_text = exp_text.replace('from datasets import load_dataset\n', '')
old_block = '''        # load data
        dataset = load_dataset("Dingdong-Inc/FreshRetailNet-50K")
        df_train = dataset['train'].to_pandas()
        df_eval = dataset['eval'].to_pandas()
'''
new_block = '''        if not self.args.train_path or not self.args.eval_path:
            raise FileNotFoundError('--train_path and --eval_path are required for local metric evaluation')
        df_train = pd.read_parquet(self.args.train_path)
        df_eval = pd.read_parquet(self.args.eval_path)
'''
if 'load_dataset("Dingdong-Inc/FreshRetailNet-50K")' in exp_text:
    exp_text = exp_text.replace(old_block, new_block)
    exp_py.write_text(exp_text, encoding='utf-8')

print('Data and metric patches applied.')

In [ ]:
# Run the 200-pair smoke test before the three full runs.
import pandas as pd, os

smoke_dir = '/content/task14_smoke'
os.makedirs(smoke_dir, exist_ok=True)
train = pd.read_parquet(TRAIN_PATH)
pairs = train[['store_id', 'product_id']].drop_duplicates().head(200)

smoke_train = train.merge(pairs, on=['store_id', 'product_id'], how='inner').sort_values(['store_id', 'product_id', 'dt'])
smoke_eval = pd.read_parquet(EVAL_PATH).merge(pairs, on=['store_id', 'product_id'], how='inner').sort_values(['store_id', 'product_id', 'dt'])
smoke_train.to_parquet(os.path.join(smoke_dir, 'train.parquet'), index=False)
smoke_eval.to_parquet(os.path.join(smoke_dir, 'eval.parquet'), index=False)

print('Smoke rows:', len(smoke_train), len(smoke_eval))

In [ ]:
%%bash
set -e
cd /content/frn-50k-baseline/demand_forecasting/DLinear
python -u main.py \
  --is_training 1 --train_only True --model DLinear \
  --scale True --loss mae --features MS --data censored_dataset \
  --data_path /content/task14_smoke/train.parquet \
  --train_path /content/task14_smoke/train.parquet \
  --eval_path /content/task14_smoke/eval.parquet \
  --target sale_amount --revin False --total_seq_len 90 --seq_len 62 \
  --pred_len 7 --moving_avg 28 --enc_in 7 --des raw_smoke \
  --do_predict --itr 1 --batch_size 1024 --train_epochs 6 \
  --learning_rate 0.001 --num_workers 2 \
  2>&1 | tee /content/task14_dlinear_smoke.log

In [ ]:
import subprocess, os

dlinear_dir = '/content/frn-50k-baseline/demand_forecasting/DLinear'

common = [
    'python', '-u', 'main.py',
    '--is_training', '1', '--train_only', 'True', '--model', 'DLinear',
    '--scale', 'True', '--loss', 'mae', '--features', 'MS',
    '--revin', 'False', '--total_seq_len', '90', '--seq_len', '62',
    '--pred_len', '7', '--moving_avg', '28', '--enc_in', '7',
    '--do_predict', '--itr', '1', '--batch_size', '1024',
    '--train_epochs', '6', '--learning_rate', '0.001', '--num_workers', '2'
]

configs = [
    {
        'name': 'DLinear_RawSales', 'des': 'raw_l1',
        'data': 'censored_dataset', 'data_path': TRAIN_PATH,
        'target': 'sale_amount', 'log': '/content/dlinear_raw_sales.log'
    },
    {
        'name': 'DLinear_TimesNet', 'des': 'timesnet_l1',
        'data': 'recovered_dataset', 'data_path': TIMESNET_PATH,
        'target': 'sale_amount_pred', 'log': '/content/dlinear_timesnet.log'
    },
    {
        'name': 'DLinear_DLinear', 'des': 'dlinear_l1',
        'data': 'recovered_dataset', 'data_path': DLINEAR_PATH,
        'target': 'sale_amount_pred', 'log': '/content/dlinear_dlinear.log'
    }
]

for config in configs:
    command = common + [
        '--data', config['data'], '--data_path', config['data_path'],
        '--train_path', TRAIN_PATH, '--eval_path', EVAL_PATH,
        '--target', config['target'], '--des', config['des']
    ]
    print(f"Running {config['name']} ...")
    with open(config['log'], 'w', encoding='utf-8') as log:
        result = subprocess.run(command, cwd=dlinear_dir, stdout=log, stderr=subprocess.STDOUT)
    if result.returncode != 0:
        raise RuntimeError(f"{config['name']} failed; see {config['log']}")
    print(f"{config['name']} exit_code=0")

In [ ]:
# Aggregate exactly 9 metric rows and verify 206,562 eligible evaluation rows.
import pandas as pd
import numpy as np
from pathlib import Path

dlinear_dir = Path('/content/frn-50k-baseline/demand_forecasting/DLinear')
paper = {
    'DLinear_RawSales': (31.56, -4.89),
    'DLinear_TimesNet': (30.73, 6.88),
    'DLinear_DLinear': (30.60, 4.02)
}
folders = {
    'DLinear_RawSales': 'DLinear_censored_dataset_ftMS_sl62_ma28_pl7_raw_l1_0',
    'DLinear_TimesNet': 'DLinear_recovered_dataset_ftMS_sl62_ma28_pl7_timesnet_l1_0',
    'DLinear_DLinear': 'DLinear_recovered_dataset_ftMS_sl62_ma28_pl7_dlinear_l1_0'
}
groups = [('psd>=0', 'Overall'), ('psd<1', 'Low Sale'), ('psd>=1', 'High Sale')]
rows = []

eligible = None
for config_id, folder in folders.items():
    metrics_path = dlinear_dir / 'results' / folder / 'metrics_res.parquet'
    metrics = pd.read_parquet(metrics_path)
    if eligible is None:
        eligible = int(round(metrics.loc[metrics.group_type == 'psd>=0', 'sample_cnt'].sum()))
    for group_id, group_name in groups:
        subset = metrics.loc[metrics.group_type == group_id]
        wape = float(subset['wape'].mean() * 100)
        wpe = float(subset['wpe'].mean() * 100)
        mae = float(subset['mae'].mean())
        paper_wape = paper_wpe = delta_wape = delta_wpe = None
        if group_name == 'Overall':
            paper_wape, paper_wpe = paper[config_id]
            delta_wape = wape - paper_wape
            delta_wpe = wpe - paper_wpe
        rows.append([
            config_id, group_name, round(wape, 4), round(wpe, 4), round(mae, 4),
            round(paper_wape, 4) if paper_wape is not None else None,
            round(paper_wpe, 4) if paper_wpe is not None else None,
            round(delta_wape, 4) if delta_wape is not None else None,
            round(delta_wpe, 4) if delta_wpe is not None else None,
            'completed'
        ])

columns = ['config_id', 'group', 'wape_pct', 'wpe_pct', 'mae',
           'paper_wape_pct', 'paper_wpe_pct', 'delta_wape_pp', 'delta_wpe_pp', 'status']
out = pd.DataFrame(rows, columns=columns)
os.makedirs('/content/task14_results', exist_ok=True)
out.to_csv('/content/task14_results/dlinear_forecasting_metrics.csv', index=False)

assert len(out) == 9
assert eligible == 206562
assert np.isfinite(out[['wape_pct', 'wpe_pct', 'mae']].to_numpy(dtype=float)).all()
print(out.to_string(index=False))
print('eligible_eval_rows_psd_gte_0 =', eligible)

In [ ]:
# Copy the final deliverables to Google Drive and save run metadata.
import json, shutil, os

drive_task = '/content/drive/MyDrive/PTDLKD/Task14_DLinear_Forecasting'
os.makedirs(drive_task + '/results/full', exist_ok=True)
os.makedirs(drive_task + '/results/smoke', exist_ok=True)
os.makedirs(drive_task + '/notebooks', exist_ok=True)

shutil.copy2('/content/task14_results/dlinear_forecasting_metrics.csv', drive_task + '/results/dlinear_forecasting_metrics.csv')
for name in ['dlinear_raw_sales.log', 'dlinear_timesnet.log', 'dlinear_dlinear.log']:
    shutil.copy2('/content/' + name, drive_task + '/results/full/' + name)
shutil.copy2('/content/task14_dlinear_smoke.log', drive_task + '/results/smoke/dlinear_smoke.log')

metadata = {
    'task': 'Task14_DLinear_Forecasting',
    'commit': '8af694a09a5a427d1d50648d348c022565a781f2',
    'seed': 2025,
    'eligible_eval_rows_psd_gte_0': 206562,
    'configs': [
        {'config_id': 'DLinear_RawSales', 'des': 'raw_l1', 'exit_code': 0, 'status': 'completed'},
        {'config_id': 'DLinear_TimesNet', 'des': 'timesnet_l1', 'exit_code': 0, 'status': 'completed'},
        {'config_id': 'DLinear_DLinear', 'des': 'dlinear_l1', 'exit_code': 0, 'status': 'completed'}
    ]
}
with open(drive_task + '/results/run_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print('Saved deliverables to:', drive_task)